# 03 — Model Training Orchestrator  —  VERSION PC (1 GPU)
## Hackathon IBM — Détection de Fraude Bancaire

Cette version est **strictement identique** à `03_Model_Training.ipynb` (workbench 4×T4) sauf :

| Paramètre | Workbench 4×T4 | **PC 1 GPU (ce fichier)** |
|---|---|---|
| `CATBOOST_ITER` | 2000 | **800** |
| `XGBOOST_ITER`  | 1500 | **800** |
| `TABNET_EPOCHS` | 80   | **40** |
| Batch size TabNet max | 1024 | **512** |
| Mode de run par défaut | `full` (10 ratios) | **`smoke` (3 ratios)** |
| Monitoring VRAM | non | **oui** (avant/après chaque modèle) |

**Objectif** : valider que le pipeline tourne bout-en-bout sur ton GPU local avant de lancer le vrai run sur le workbench.

**Après validation**, tu peux passer `RUN_MODE = 'full'` pour lancer les 10 ratios sur ton PC (plus lent) ou basculer sur le notebook workbench.

---
## 0. Installation (à exécuter une fois)

La cellule ci-dessous installe tout ce qu'il faut. Si `torch` est déjà installé en CPU-only sur ton PC (ce qui est le cas par défaut), on **réinstalle** une build CUDA compatible.

> **Vérifie d'abord ta version CUDA** : `nvidia-smi` dans un terminal → colonne *CUDA Version*. Si tu as CUDA 12.1+, garde `cu121`. Si tu as CUDA 11.8, remplace par `cu118`.

In [ ]:
# Si déjà installé, cette cellule est idempotente et termine en <5s.
import subprocess, sys

def pip_install(args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# 1) Libs de base (hors torch)
pip_install(['catboost', 'xgboost', 'pytorch-tabnet', 'scikit-learn',
             'pandas', 'numpy', 'pyarrow', 'joblib', 'tqdm', 'matplotlib'])

# 2) torch + CUDA : décommente la ligne qui correspond à TA version CUDA (vérifier via `nvidia-smi`)
# pip_install(['torch', '--index-url', 'https://download.pytorch.org/whl/cu121'])  # CUDA 12.1+
# pip_install(['torch', '--index-url', 'https://download.pytorch.org/whl/cu118'])  # CUDA 11.8
# pip_install(['torch'])                                                             # CPU-only fallback

print('Installation OK — redémarre le kernel si c\'est la première installation de torch avec CUDA.')

---
## 1. Imports & détection du matériel

In [ ]:
import os
import re
import gc
import time
import glob
import json
import warnings
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import catboost as cb
import xgboost as xgb

import torch
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

CUDA_AVAILABLE = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if CUDA_AVAILABLE else 0
DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'
XGB_VERSION = tuple(int(x) for x in xgb.__version__.split('.')[:2])

print('=== Environment ===')
print(f'pandas   : {pd.__version__}')
print(f'numpy    : {np.__version__}')
print(f'torch    : {torch.__version__}')
print(f'catboost : {cb.__version__}')
print(f'xgboost  : {xgb.__version__}')
print(f'CUDA     : {CUDA_AVAILABLE} (devices={N_GPUS})')
if CUDA_AVAILABLE:
    for i in range(N_GPUS):
        props = torch.cuda.get_device_properties(i)
        total_vram = props.total_memory / 1024**3
        print(f'  GPU[{i}] : {props.name}  |  VRAM total : {total_vram:.1f} GB')
else:
    print('\n⚠ Aucune CUDA détectée → fallback CPU. L\'entraînement sera plus lent.')
    print('  Pour utiliser ton GPU, installe la bonne version de torch (cf. cellule 0).')


def print_vram(tag: str = ''):
    """Affiche la consommation VRAM courante (utile pour debug single-GPU)."""
    if not CUDA_AVAILABLE:
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    print(f'    VRAM {tag:<12} allocated={allocated:.2f} GB | reserved={reserved:.2f} GB')

---
## 2. Configuration globale (budget PC)

**`RUN_MODE`** :
- `'smoke'` (défaut) : 3 ratios représentatifs (2 %, 10 %, 30 %) — **~2-5 min** sur GPU milieu de gamme.
- `'full'` : les 10 ratios — **~15-30 min** selon ton GPU.

In [ ]:
DATA_DIR    = Path('.')
TRAIN_GLOB  = 'prepared_train_*.parquet'
TEST_FILE   = 'prepared_test_050.0_pct.parquet'

LOG_DIR     = Path('logs_training_pc')
MODELS_DIR  = LOG_DIR / 'models'
RESULTS_CSV = LOG_DIR / 'experiment_results.csv'

LOG_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)

TARGET = 'is_fraud'
RANDOM_STATE = 42

# --- Budget PC (réduit par rapport au workbench) ---
CATBOOST_ITER       = 800
XGBOOST_ITER        = 800
TABNET_EPOCHS       = 40
TABNET_BATCH_MAX    = 512       # borne supérieure du batch size sur PC

# --- Mode de run : 'smoke' = 3 ratios, 'full' = 10 ratios ---
RUN_MODE = 'smoke'
SMOKE_RATIOS = [2.0, 10.0, 30.0]   # représentatif : bas / moyen / haut

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f'Run mode    : {RUN_MODE}')
print(f'Logs        -> {LOG_DIR.resolve()}')
print(f'Models      -> {MODELS_DIR.resolve()}')
print(f'Results CSV -> {RESULTS_CSV.resolve()}')

---
## 3. Helpers génériques

In [ ]:
_RATIO_RE = re.compile(r'(\d+\.\d+)')


def extract_ratio(path) -> float:
    """'prepared_train_010.0_pct.parquet' -> 10.0"""
    m = _RATIO_RE.search(Path(path).stem)
    return float(m.group(1)) if m else np.nan


def load_dataset(path) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """Charge un .parquet préparé. Retourne (X, y, cat_feature_names)."""
    df = pd.read_parquet(path)
    y = df[TARGET].astype(np.int8)
    X = df.drop(columns=[TARGET])
    cat_features = [c for c, dt in X.dtypes.items() if str(dt) == 'category']
    return X, y, cat_features


def align_categories(X_train: pd.DataFrame, X_test: pd.DataFrame,
                      cat_features: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Force les mêmes catégories dans train et test (union des modalités)."""
    X_train = X_train.copy()
    X_test  = X_test.copy()
    for c in cat_features:
        all_cats = (
            pd.api.types.union_categoricals([X_train[c], X_test[c]], sort_categories=True)
              .categories
        )
        X_train[c] = pd.Categorical(X_train[c], categories=all_cats)
        X_test[c]  = pd.Categorical(X_test[c],  categories=all_cats)
    return X_train, X_test


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray) -> Dict[str, float]:
    """Calcule toutes les métriques de fraude + matrice de confusion."""
    metrics = {
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'PR-AUC':    average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'ROC-AUC':   roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0
    metrics.update({'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)})
    return metrics


def append_result_row(row: dict, csv_path: Path) -> None:
    df_row = pd.DataFrame([row])
    header = not csv_path.exists()
    df_row.to_csv(csv_path, mode='a', header=header, index=False)

---
## 4. Entraîneur — CatBoost (GPU 0 sur PC)

In [ ]:
def train_catboost(X_train, y_train, X_test, y_test, cat_features) -> Tuple:
    Xtr = X_train.copy()
    Xte = X_test.copy()
    for c in cat_features:
        Xtr[c] = Xtr[c].astype(str).fillna('missing')
        Xte[c] = Xte[c].astype(str).fillna('missing')

    params = dict(
        iterations=CATBOOST_ITER,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE,
        eval_metric='PRAUC',
        loss_function='Logloss',
        auto_class_weights='Balanced',
        od_type='Iter',
        od_wait=80,
        verbose=False,
        allow_writing_files=False,
    )
    if CUDA_AVAILABLE:
        # devices='0' : un seul GPU, explicite pour éviter les conflits sur PC multi-GPU
        params.update(dict(task_type='GPU', devices='0'))
    else:
        params.update(dict(task_type='CPU', thread_count=-1))

    model = cb.CatBoostClassifier(**params)

    t0 = time.perf_counter()
    model.fit(
        Xtr, y_train,
        cat_features=cat_features,
        eval_set=(Xte, y_test),
        use_best_model=True,
    )
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(Xte)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_catboost(model, path: Path) -> None:
    model.save_model(str(path.with_suffix('.cbm')))

---
## 5. Entraîneur — XGBoost (GPU 0 sur PC)

In [ ]:
def train_xgboost(X_train, y_train, X_test, y_test, cat_features) -> Tuple:
    X_train_al, X_test_al = align_categories(X_train, X_test, cat_features)

    params = dict(
        n_estimators=XGBOOST_ITER,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        reg_alpha=0.0,
        objective='binary:logistic',
        eval_metric='aucpr',
        scale_pos_weight=float((y_train == 0).sum() / max((y_train == 1).sum(), 1)),
        random_state=RANDOM_STATE,
        enable_categorical=True,
        n_jobs=-1,
        early_stopping_rounds=80,
        verbosity=0,
    )
    if CUDA_AVAILABLE:
        if XGB_VERSION >= (2, 0):
            # device='cuda:0' : force explicitement le premier GPU
            params.update(dict(tree_method='hist', device='cuda:0'))
        else:
            params.update(dict(tree_method='gpu_hist', gpu_id=0, predictor='gpu_predictor'))
    else:
        params.update(dict(tree_method='hist'))

    model = xgb.XGBClassifier(**params)

    t0 = time.perf_counter()
    model.fit(
        X_train_al, y_train,
        eval_set=[(X_test_al, y_test)],
        verbose=False,
    )
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test_al)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_xgboost(model, path: Path) -> None:
    model.save_model(str(path.with_suffix('.ubj')))

---
## 6. Entraîneur — TabNet (cuda:0, batch size borné)

In [ ]:
def build_tabnet_preprocessing(X_train: pd.DataFrame, X_test: pd.DataFrame,
                                cat_features: List[str]) -> dict:
    num_cols = [c for c in X_train.columns if c not in cat_features]
    ordered_cols = cat_features + num_cols
    X_tr = X_train[ordered_cols].copy()
    X_te = X_test[ordered_cols].copy()

    cat_dims = []
    for c in cat_features:
        enc = LabelEncoder()
        combined = pd.concat([X_tr[c].astype(str), X_te[c].astype(str)], axis=0)
        enc.fit(combined.fillna('missing'))
        X_tr[c] = enc.transform(X_tr[c].astype(str).fillna('missing'))
        X_te[c] = enc.transform(X_te[c].astype(str).fillna('missing'))
        cat_dims.append(len(enc.classes_))

    cat_idxs = list(range(len(cat_features)))

    imputer = SimpleImputer(strategy='median')
    scaler  = StandardScaler()

    X_tr[num_cols] = imputer.fit_transform(X_tr[num_cols])
    X_te[num_cols] = imputer.transform(X_te[num_cols])
    X_tr[num_cols] = scaler.fit_transform(X_tr[num_cols])
    X_te[num_cols] = scaler.transform(X_te[num_cols])

    return {
        'X_train_np': X_tr.values.astype(np.float32),
        'X_test_np':  X_te.values.astype(np.float32),
        'cat_idxs':   cat_idxs,
        'cat_dims':   cat_dims,
    }


def train_tabnet(X_train, y_train, X_test, y_test, cat_features) -> Tuple:
    prep = build_tabnet_preprocessing(X_train, X_test, cat_features)

    n = len(X_train)
    # Batch borné par TABNET_BATCH_MAX pour respecter la VRAM d'un GPU de PC
    batch_size         = min(TABNET_BATCH_MAX, max(32, 2 ** int(np.log2(max(n // 8, 32)))))
    virtual_batch_size = min(128, max(16, batch_size // 8))

    model = TabNetClassifier(
        cat_idxs=prep['cat_idxs'],
        cat_dims=prep['cat_dims'],
        cat_emb_dim=4,
        n_d=16, n_a=16, n_steps=3,
        gamma=1.3, lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params=dict(step_size=15, gamma=0.9),
        seed=RANDOM_STATE,
        device_name='cuda' if CUDA_AVAILABLE else 'cpu',
        verbose=0,
    )

    t0 = time.perf_counter()
    model.fit(
        X_train=prep['X_train_np'], y_train=y_train.values.astype(np.int64),
        eval_set=[(prep['X_test_np'], y_test.values.astype(np.int64))],
        eval_name=['test'],
        eval_metric=['auc'],
        max_epochs=TABNET_EPOCHS,
        patience=15,
        batch_size=batch_size,
        virtual_batch_size=virtual_batch_size,
        num_workers=0,
        drop_last=False,
        weights=1,
    )
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(prep['X_test_np'])[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_tabnet(model, path: Path) -> None:
    model.save_model(str(path))

---
## 7. Registre des modèles

In [ ]:
MODEL_REGISTRY = {
    'CatBoost': dict(train_fn=train_catboost, save_fn=save_catboost),
    'XGBoost':  dict(train_fn=train_xgboost,  save_fn=save_xgboost),
    'TabNet':   dict(train_fn=train_tabnet,   save_fn=save_tabnet),
}

print('Modèles enregistrés :', list(MODEL_REGISTRY.keys()))

---
## 8. Chargement unique du test set

In [ ]:
test_path = DATA_DIR / TEST_FILE
X_test, y_test, cat_features_test = load_dataset(test_path)

print(f'Test set  : {test_path.name}')
print(f'  shape   : {X_test.shape}')
print(f'  fraude  : {y_test.mean()*100:.3f} %  ({int(y_test.sum())} positifs / {len(y_test)})')
print(f'  cat cols: {len(cat_features_test)}')
print(f'  mémoire : {X_test.memory_usage(deep=True).sum()/1024**2:.2f} MB')

---
## 9. Boucle d'expérimentation principale

**Monitoring VRAM** affiché avant/après chaque modèle pour surveiller le remplissage GPU.

In [ ]:
def run_single_experiment(train_path: Path, model_name: str, cfg: dict,
                           X_test, y_test) -> dict:
    ratio = extract_ratio(train_path)
    X_train, y_train, cat_features = load_dataset(train_path)

    print(f'  ├─ {model_name:<8} | train={len(X_train):>6,} | '
          f'fraud_train={y_train.mean()*100:>5.2f}%')
    print_vram(f'[{model_name}] pre')

    t_global = time.perf_counter()
    model, y_pred, y_proba, train_time = cfg['train_fn'](
        X_train, y_train, X_test, y_test, cat_features
    )
    print_vram(f'[{model_name}] post')

    metrics = compute_metrics(y_test.values, y_pred, y_proba)

    ratio_tag  = f'{ratio:05.1f}'.replace('.', '_')
    model_path = MODELS_DIR / f'{model_name.lower()}_train_{ratio_tag}'
    cfg['save_fn'](model, model_path)

    row = {
        'Model':                 model_name,
        'Dataset_Ratio':         ratio,
        'Dataset_File':          Path(train_path).name,
        'N_Train':               len(X_train),
        'Fraud_Rate_Train_%':    round(y_train.mean() * 100, 4),
        **{k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()},
        'Training_Time_Seconds': round(train_time, 2),
        'Total_Time_Seconds':    round(time.perf_counter() - t_global, 2),
    }
    append_result_row(row, RESULTS_CSV)

    print(f'    └─ F1={metrics["F1"]:.3f} | PR-AUC={metrics["PR-AUC"]:.3f} | '
          f'Recall={metrics["Recall"]:.3f} | time={train_time:6.1f}s')

    del model, X_train, y_train, y_pred, y_proba
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

    return row

In [ ]:
if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

# Sélection des fichiers selon le RUN_MODE
all_train_files = sorted(DATA_DIR.glob(TRAIN_GLOB))

if RUN_MODE == 'smoke':
    train_files = [f for f in all_train_files if extract_ratio(f) in SMOKE_RATIOS]
    print(f'=== SMOKE MODE : {len(train_files)} fichier(s) ciblé(s) (ratios {SMOKE_RATIOS}) ===')
else:
    train_files = all_train_files
    print(f'=== FULL MODE : {len(train_files)} fichier(s) ===')

for f in train_files:
    print(f'  - {f.name:<40} (ratio={extract_ratio(f):.1f}%)')
print()

t_all = time.perf_counter()
all_results = []

for i, train_path in enumerate(train_files, 1):
    ratio = extract_ratio(train_path)
    print(f'[{i}/{len(train_files)}] {train_path.name}  (ratio={ratio:.1f}%)')

    for model_name, cfg in MODEL_REGISTRY.items():
        try:
            row = run_single_experiment(train_path, model_name, cfg, X_test, y_test)
            all_results.append(row)
        except Exception as e:
            print(f'  ├─ {model_name:<8} | ❌ ERREUR : {type(e).__name__}: {e}')
            append_result_row({
                'Model': model_name,
                'Dataset_Ratio': ratio,
                'Dataset_File': train_path.name,
                'Error': f'{type(e).__name__}: {e}',
            }, RESULTS_CSV)
    print()

print(f'=== TERMINÉ en {(time.perf_counter()-t_all)/60:.1f} min ===')
print(f'Résultats sauvegardés dans : {RESULTS_CSV}')

---
## 10. Analyse des résultats

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Résultats : {results.shape}')
results

In [ ]:
if 'PR-AUC' in results.columns:
    lb = (
        results
        .dropna(subset=['PR-AUC'])
        .sort_values('PR-AUC', ascending=False)
        .loc[:, ['Model', 'Dataset_Ratio', 'F1', 'Recall', 'Precision',
                 'PR-AUC', 'ROC-AUC', 'TP', 'FP', 'FN', 'TN',
                 'Training_Time_Seconds']]
        .reset_index(drop=True)
    )
    print('=== TOP par PR-AUC ===')
    print(lb.head(10).to_string(index=False))

    print('\n=== Meilleur résultat par modèle ===')
    best_per_model = lb.loc[lb.groupby('Model')['PR-AUC'].idxmax()].sort_values('PR-AUC', ascending=False)
    print(best_per_model.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ['F1', 'Recall', 'Precision', 'PR-AUC']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, m in zip(axes.ravel(), metrics_to_plot):
    for model_name in results['Model'].dropna().unique():
        sub = (results[results['Model'] == model_name]
               .dropna(subset=[m])
               .sort_values('Dataset_Ratio'))
        ax.plot(sub['Dataset_Ratio'], sub[m], marker='o', lw=2, label=model_name)
    ax.set_xlabel('Ratio undersampling train (% de fraude)')
    ax.set_ylabel(m)
    ax.set_title(f'{m} vs ratio d\'undersampling')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(LOG_DIR / 'metrics_vs_ratio.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 11. Bilan & passage au workbench

**Si le smoke test s'est bien passé** :
1. Le pipeline est validé de bout-en-bout sur ton GPU → aucune erreur de config.
2. Les modèles se chargent, s'entraînent, et sauvegardent correctement.
3. Le CSV est bien formaté.

**Deux options ensuite** :
- **Option A (PC uniquement)** : passer `RUN_MODE = 'full'` et relancer → les 10 ratios tournent sur ton GPU (15-30 min).
- **Option B (recommandée)** : transférer les parquets + le notebook `03_Model_Training.ipynb` (workbench, budget plein) sur la machine 4×T4 et relancer là-bas. Les 4 GPUs ne sont pas utilisés en parallèle par ce notebook (les libs utilisent un GPU chacune), mais le budget plus élevé (2000/1500/80 itérations) et les T4 feront la différence en qualité.